## Ch1. Getting Started Forecasting: Principles & Practice (Python Edition) Extracted from: fpppy-01-intro.qmd

## [Setup] Imports & Configuration --- Setup / Hidden in slides ---

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os; os.environ["NIXTLA_ID_AS_COL"] = "true"
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import Naive, SeasonalNaive, AutoETS, AutoARIMA
plt.rcParams.update({"figure.figsize": (7, 3.5)})

## [Slide 11] 1.7 Point Forecasts vs.\ Distributions

In [ ]:
import numpy as np, matplotlib.pyplot as plt

np.random.seed(42)
t = np.arange(1, 25)
y = 10 + 0.5*t + 2*np.sin(2*np.pi*t/4) + np.random.randn(24)

h = np.arange(25, 33)
mu = 10 + 0.5*h + 2*np.sin(2*np.pi*h/4)
sigma = 1.5 * np.sqrt(h - 24)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(t, y, 'b-o', ms=4, label='Observed')
ax.plot(h, mu, 'r--', label='Forecast')
ax.fill_between(h, mu-1.28*sigma, mu+1.28*sigma, alpha=0.3, color='red', label='80% PI')
ax.fill_between(h, mu-1.96*sigma, mu+1.96*sigma, alpha=0.15, color='red', label='95% PI')
ax.axvline(24.5, color='gray', linestyle=':', lw=1)
ax.legend(fontsize=8); ax.set_xlabel('Time'); ax.set_ylabel('Value')
plt.tight_layout(); plt.show()

## [Slide 14] 1.8 Installing the Libraries Install (run once in terminal) pip install statsforecast neuralforecast hierarchicalforecast pip install mlforecast utilsforecast

Core imports

In [ ]:
import pandas as pd
import numpy as np
from statsforecast import StatsForecast
from statsforecast.models import (
    Naive,
    SeasonalNaive,
    AutoETS,
    AutoARIMA,
    HistoricAverage,
)
from utilsforecast.plotting import plot_series
from utilsforecast.losses import mae, smape, mase

## [Slide 16] 1.9 A First Forecasting Example: Data Structure

In [ ]:
import pandas as pd
import numpy as np

Nixtlaverse format: long DataFrame with (unique_id, ds, y)

In [ ]:
np.random.seed(0)
dates = pd.date_range("2018-01", periods=60, freq="MS")
y = (100 + np.arange(60)*0.5
     + 10*np.sin(2*np.pi*np.arange(60)/12)
     + np.random.randn(60)*3)

df = pd.DataFrame({
    "unique_id": "series_1",
    "ds": dates,
    "y": y
})
print(df.head())

unique_id         ds          y
0  series_1 2018-01-01  100.15
1  series_1 2018-02-01  101.67
...

## [Slide 18] 1.9 A First Forecasting Example: Fit \& Forecast

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoARIMA, SeasonalNaive

1. Instantiate models

In [ ]:
models = [
    AutoETS(season_length=12),
    AutoARIMA(season_length=12),
    SeasonalNaive(season_length=12),
]

2. Create StatsForecast object

In [ ]:
sf = StatsForecast(models=models, freq="MS", n_jobs=-1)

3. Fit and forecast (h = 12 months ahead)

In [ ]:
fcst = sf.forecast(df=df, h=12, level=[80, 95])
print(fcst.head())

## [Slide 20] 1.9 A First Forecasting Example: Evaluate

In [ ]:
from utilsforecast.losses import mae, mase, smape
from utilsforecast.evaluation import evaluate

Cross-validation for honest evaluation

In [ ]:
cv = sf.cross_validation(
    df=df,
    h=12,
    step_size=6,
    n_windows=3,
)

Compute error metrics

In [ ]:
errors = evaluate(
    cv,
    metrics=[mae, smape, mase],
    train_df=df,
)
print(errors)